# Probe accuracy by layer

A linear probe asks what a layer makes linearly readable: train a linear classifier
on the layer's activations and measure its held-out accuracy. Plotting that accuracy against depth
shows where a network builds a feature and where it throws one away.

Papers:

- Guillaume Alain and Yoshua Bengio (2016), "Understanding intermediate layers using linear
  classifier probes", https://arxiv.org/abs/1610.01644
- Yonatan Belinkov (2022), "Probing Classifiers: Promises, Shortcomings, and Advances",
  Computational Linguistics 48(1), https://doi.org/10.1162/coli_a_00422

TorchLens makes this a few lines: one `tl.trace(..., save="all")` records every operation's output
(including functional calls such as `F.relu` and `F.max_pool2d`, which module hooks miss), and the
probe loop walks `trace.layer_labels` in execution order.

The model is a tiny CNN trained here on generated images, so the notebook runs offline on a laptop
CPU in seconds: each image holds one of four shapes in one of four quadrants, and the network is
trained to name the shape only. We probe every layer for the shape (the trained label) and for the
quadrant (a property it was never asked about), and repeat both on an untrained copy as the control
Belinkov recommends: a probe can read structure that the architecture alone provides.

In [ ]:
# recipe-test: tool=matplotlib expected=True
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

import torchlens as tl


def make_shapes(n: int, seed: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Noisy 16x16 images of one of four shapes in one of four quadrants."""
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(7) - 3, torch.arange(7) - 3, indexing="ij")
    stamps = torch.stack(
        [
            (yy.abs().maximum(xx.abs()) == 3).float(),  # square outline
            (yy**2 + xx**2 <= 9).float(),  # disc
            ((yy == 0) | (xx == 0)).float(),  # plus
            (yy.abs() == xx.abs()).float(),  # cross
        ]
    )
    shape = torch.randint(0, 4, (n,), generator=g)
    top, left = torch.randint(0, 10, (2, n), generator=g)  # anywhere on the canvas
    images = 0.3 * torch.randn(n, 1, 16, 16, generator=g)
    for i in range(n):
        images[i, 0, top[i] : top[i] + 7, left[i] : left[i] + 7] += stamps[shape[i]]
    quadrant = 2 * (top >= 5) + (left >= 5)  # which quadrant holds the shape's centre
    return images, shape, quadrant


class TinyCNN(nn.Module):
    """Three conv stages, global average pooling, a linear shape classifier."""

    def __init__(self, width: int = 8) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, width, 3, padding=1)
        self.conv2 = nn.Conv2d(width, 2 * width, 3, padding=1)
        self.conv3 = nn.Conv2d(2 * width, 4 * width, 3, padding=1)
        self.head = nn.Linear(4 * width, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.conv3(x)).mean(dim=(2, 3))  # position is pooled away here
        return self.head(x)


def train(
    model: nn.Module, images: torch.Tensor, labels: torch.Tensor, epochs: int = 12
) -> nn.Module:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        for batch in torch.randperm(len(images)).split(64):
            optimizer.zero_grad()
            F.cross_entropy(model(images[batch]), labels[batch]).backward()
            optimizer.step()
    return model.eval()


train_images, train_shape, _ = make_shapes(1000, seed=0)

torch.manual_seed(0)
untrained = TinyCNN().eval()
torch.manual_seed(0)
trained = train(TinyCNN(), train_images, train_shape)

test_images, test_shape, _ = make_shapes(400, seed=1)
accuracy = (trained(test_images).argmax(1) == test_shape).float().mean().item()
RESULT = accuracy > 0.9

## 1. One trace, every layer

Trace a fresh probe set once per network and keep each layer's output. The input itself is
layer 0, the pixel baseline.

In [ ]:
# recipe-test: tool=matplotlib expected=True
probe_images, probe_shape, probe_quadrant = make_shapes(800, seed=2)


def layer_outputs(model: nn.Module) -> dict[str, torch.Tensor]:
    """Every layer's output from one trace, in execution order, one row per image."""
    trace = tl.trace(model, probe_images, save="all")
    return {
        label: trace[label].out.detach().flatten(1)
        for label in trace.layer_labels
        if label not in trace.output_layers  # the output alias repeats the last layer
    }


trained_layers, untrained_layers = layer_outputs(trained), layer_outputs(untrained)
print(list(trained_layers))
RESULT = list(trained_layers) == list(untrained_layers) and len(trained_layers) >= 10

## 2. Train a linear probe on each layer

Each probe is a ridge-regression classifier on standardized activations: fit to one-hot labels on
500 images in closed form, scored by argmax on the other 300. It is the least-squares cousin of
logistic regression and fits every layer of both networks in about a second.

In [ ]:
# recipe-test: tool=matplotlib expected=True
def probe_accuracy(features: torch.Tensor, labels: torch.Tensor) -> float:
    train_rows, test_rows = slice(0, 500), slice(500, None)
    mean, std = features[train_rows].mean(0), features[train_rows].std(0) + 1e-6
    x = ((features - mean) / std).double()
    train_x, test_x = x[train_rows], x[test_rows]
    targets = F.one_hot(labels[train_rows], 4).double()
    # Ridge in its dual form (one 500 x 500 solve), penalty scaled to the feature count.
    gram = train_x @ train_x.T + x.shape[1] * torch.eye(len(train_x), dtype=torch.double)
    scores = test_x @ train_x.T @ torch.linalg.solve(gram, targets)
    return (scores.argmax(1) == labels[test_rows]).float().mean().item()


curves = {
    (net, target): [probe_accuracy(acts, labels) for acts in layers.values()]
    for net, layers in [("trained", trained_layers), ("untrained", untrained_layers)]
    for target, labels in [("shape", probe_shape), ("quadrant", probe_quadrant)]
}
for key, values in curves.items():
    print(key, [round(v, 2) for v in values])
shape, quadrant = curves["trained", "shape"], curves["trained", "quadrant"]
pooled = list(trained_layers).index(next(k for k in trained_layers if k.startswith("mean")))
untrained_shape = curves["untrained", "shape"]
RESULT = (
    shape[-1] > shape[0] + 0.2
    and shape[-1] > untrained_shape[-1]
    and quadrant[pooled] < quadrant[0] - 0.2
)

## 3. Plot accuracy against depth

Shape becomes linearly readable with depth in the trained network and ends far above the untrained
control. The quadrant is readable from the pixels and early layers and falls at the global average
pool: the network keeps only what its task needs. Chance is 0.25.

In [ ]:
# recipe-test: tool=matplotlib expected=True
labels = list(trained_layers)
fig, ax = plt.subplots(figsize=(8, 3.5))
for (net, target), values in curves.items():
    ax.plot(
        values,
        marker="o",
        color="tab:blue" if target == "shape" else "tab:orange",
        linestyle="-" if net == "trained" else "--",
        label=f"{target}, {net} network",
    )
ax.axhline(0.25, color="grey", linewidth=0.8, linestyle=":")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_ylabel("held-out probe accuracy")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("probe_accuracy_by_layer.png", dpi=150)
RESULT = len(ax.lines) == 5